# Concours de prédiction Kaggle

## Setup PyTorch GPU
Cette cellule installe (ou met à jour) PyTorch, torchvision et torchaudio avec les wheels CUDA 12.4 pour Windows.

In [1]:
import sys
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "torch",
    "torchvision",
    "torchaudio",
    "--index-url",
    "https://download.pytorch.org/whl/cu124",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Installation finished.")

Running: c:\Users\thiev\OneDrive\Bureau\Plankton\.venv\Scripts\python.exe -m pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
Installation finished.


## Validation GPU PyTorch
Cette cellule vérifie la disponibilité CUDA, lance un calcul sur GPU et exécute un mini entraînement.

In [2]:
import time
import torch
import torch.nn as nn

print("torch version:", torch.__version__)
print("compiled CUDA version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible dans ce kernel. Vérifie le kernel Python ou les drivers NVIDIA.")

device = torch.device("cuda")
print("device name:", torch.cuda.get_device_name(0))

# 1) Test de calcul matrix-multiplication sur GPU
a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)
torch.cuda.synchronize()
t0 = time.time()
c = a @ b
torch.cuda.synchronize()
print(f"matmul done on {c.device} in {time.time() - t0:.3f}s")

# 2) Mini entraînement
model = nn.Sequential(
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

x = torch.randn(2048, 128, device=device)
y = torch.randint(0, 10, (2048,), device=device)

for step in range(20):
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

print("training OK, final loss:", float(loss.detach().cpu()))
print("✅ Environnement PyTorch GPU opérationnel dans le notebook.")

torch version: 2.6.0+cu124
compiled CUDA version: 12.4
cuda available: True
device count: 1
device name: NVIDIA GeForce RTX 4060 Laptop GPU
matmul done on cuda:0 in 0.197s
training OK, final loss: 2.0091753005981445
✅ Environnement PyTorch GPU opérationnel dans le notebook.


In [3]:
# Mini réseau de neurones sur GPU (démo rapide)
import time
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "CUDA non disponible dans ce kernel"
device = torch.device("cuda")

# Données synthétiques
n_samples = 8192
n_features = 64
n_classes = 5

X = torch.randn(n_samples, n_features, device=device)
true_w = torch.randn(n_features, n_classes, device=device)
true_b = torch.randn(n_classes, device=device)
logits_true = X @ true_w + true_b
y = logits_true.argmax(dim=1)

# Modèle simple
model = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Linear(128, n_classes)
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss()

# Entraînement rapide
start = time.time()
for epoch in range(30):
    opt.zero_grad(set_to_none=True)
    out = model(X)
    loss = crit(out, y)
    loss.backward()
    opt.step()

elapsed = time.time() - start
pred = out.argmax(dim=1)
acc = (pred == y).float().mean().item()

print("device:", torch.cuda.get_device_name(0))
print(f"loss final: {loss.item():.4f}")
print(f"accuracy train: {acc*100:.2f}%")
print(f"temps total: {elapsed:.3f}s")
print("✅ Mini entraînement GPU exécuté avec succès")

device: NVIDIA GeForce RTX 4060 Laptop GPU
loss final: 0.7456
accuracy train: 83.85%
temps total: 0.045s
✅ Mini entraînement GPU exécuté avec succès


In [ ]:
# Cellule 7 — Chargement des datasets Kaggle (train/test) prêts pour PyTorch
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import datasets, transforms

DATA_ROOT = Path(".")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

assert TRAIN_DIR.exists(), f"Dossier introuvable: {TRAIN_DIR.resolve()}"
assert TEST_DIR.exists(), f"Dossier introuvable: {TEST_DIR.resolve()}"

IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.15), ratio=(0.3, 3.3), value="random"),
])

val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def _test_sort_key(path_obj):
    stem = path_obj.stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)


class TestImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = sorted(list(self.root_dir.glob("*.jpg")), key=_test_sort_key)
        if len(self.image_paths) == 0:
            self.image_paths = sorted(list(self.root_dir.rglob("*.jpg")), key=_test_sort_key)
        self.imgnames = [p.name for p in self.image_paths]  # ex: "123.jpg"

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self.image_paths[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.imgnames[index]


train_ds_full = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds_full = datasets.ImageFolder(TRAIN_DIR, transform=val_tf)
test_ds = TestImageDataset(TEST_DIR, transform=val_tf)

class_names = train_ds_full.classes
class_to_idx = train_ds_full.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print(f"Train images: {len(train_ds_full):,}")
print(f"Test images:  {len(test_ds):,}")
print(f"Nb classes:   {len(class_names)}")
print("Exemples classes:", class_names[:10])

Train images: 1,093,785
Test images:  121,428
Nb classes:   86
Exemples classes: ['000_Physonectae', '001_tail_Appendicularia', '002_other_Siphonophorae', '003_Harpacticoida', '004_empty_Harpacticoida', '005_comb_Ctenophora', '006_larvae_Echinodermata', '007_Noctiluca', '008_detritus', '009_empty_Copepoda']


In [ ]:
# Cellule 8 — Entraînement d'un CNN profond sur GPU avec suivi tqdm + macro-F1 (version full)
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import models
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    raise RuntimeError("GPU non disponible. Vérifie le kernel et CUDA.")

assert "train_ds_full" in globals(), "Exécute d'abord la cellule 7 (chargement datasets)."

# -------- CONFIG --------
FAST_MODE = False
EPOCHS = 2 if FAST_MODE else 20
MAX_TRAIN_SAMPLES = 20_000 if FAST_MODE else None
MAX_VAL_SAMPLES = 4_000 if FAST_MODE else None
BATCH_SIZE = 48
NUM_WORKERS = 4
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.10
GRAD_ACCUM_STEPS = 1
EARLY_STOPPING_PATIENCE = 6
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Ajouts simples pour améliorer la perf
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.3
MAX_GRAD_NORM = 1.0
# ------------------------

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Split stratifié par classe (plus fiable pour macro-F1)
class_to_indices = defaultdict(list)
for sample_idx, (_, class_idx) in enumerate(train_ds_full.samples):
    class_to_indices[class_idx].append(sample_idx)

train_indices = []
val_indices = []
for class_idx, idxs in class_to_indices.items():
    random.shuffle(idxs)
    n_val = max(1, int(len(idxs) * VAL_RATIO))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

random.shuffle(train_indices)
random.shuffle(val_indices)

if MAX_TRAIN_SAMPLES is not None:
    train_indices = train_indices[:MAX_TRAIN_SAMPLES]
if MAX_VAL_SAMPLES is not None:
    val_indices = val_indices[:MAX_VAL_SAMPLES]

train_subset = Subset(train_ds_full, train_indices)
val_subset = Subset(val_ds_full, val_indices)

print(f"Train subset: {len(train_subset):,} | Val subset: {len(val_subset):,}")

# Pondération des classes via sampler
train_targets = [train_ds_full.samples[i][1] for i in train_indices]
class_counts = Counter(train_targets)
class_weights = {cls: 1.0 / max(1, count) for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

num_classes = len(class_names)
weights = models.EfficientNet_B2_Weights.DEFAULT
model = models.efficientnet_b2(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
scaler = torch.amp.GradScaler("cuda")


def macro_f1_score_torch(y_true, y_pred, n_classes):
    cm = torch.zeros((n_classes, n_classes), dtype=torch.float64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1.0

    tp = torch.diag(cm)
    fp = cm.sum(dim=0) - tp
    fn = cm.sum(dim=1) - tp

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return f1.mean().item()


def maybe_mixup(images, targets, alpha, prob):
    if alpha <= 0 or random.random() > prob:
        return images, targets, None, 1.0
    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(images.size(0), device=images.device)
    mixed_images = lam * images + (1.0 - lam) * images[index]
    targets_b = targets[index]
    return mixed_images, targets, targets_b, lam


best_val_f1 = -1.0
best_state = None
best_epoch = -1
no_improve_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False)
    optimizer.zero_grad(set_to_none=True)

    for step, (images, targets) in enumerate(train_bar, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        images, targets_a, targets_b, lam = maybe_mixup(images, targets, MIXUP_ALPHA, MIXUP_PROB)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            if targets_b is None:
                loss = criterion(logits, targets_a) / GRAD_ACCUM_STEPS
            else:
                loss_a = criterion(logits, targets_a)
                loss_b = criterion(logits, targets_b)
                loss = (lam * loss_a + (1.0 - lam) * loss_b) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * images.size(0) * GRAD_ACCUM_STEPS
        train_bar.set_postfix(loss=f"{(loss.item() * GRAD_ACCUM_STEPS):.4f}")

    train_loss = running_loss / len(train_subset)

    model.eval()
    val_loss_sum = 0.0
    all_preds = []
    all_targets = []

    with torch.inference_mode():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]", leave=False)
        for images, targets in val_bar:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, targets)

            preds = torch.argmax(logits, dim=1)
            val_loss_sum += loss.item() * images.size(0)
            all_preds.append(preds.cpu())
            all_targets.append(targets.cpu())

    val_loss = val_loss_sum / len(val_subset)
    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_targets)
    val_macro_f1 = macro_f1_score_torch(y_true, y_pred, num_classes)

    scheduler.step()

    epoch_ckpt = CHECKPOINT_DIR / f"epoch_{epoch:02d}_f1_{val_macro_f1:.4f}.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "class_to_idx": class_to_idx,
            "img_size": IMG_SIZE,
            "epoch": epoch,
            "val_macro_f1": val_macro_f1,
            "fast_mode": FAST_MODE,
        },
        epoch_ckpt,
    )

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_macro_f1:.4f}"
    )

    if (not FAST_MODE) and (no_improve_epochs >= EARLY_STOPPING_PATIENCE):
        print(f"Early stopping déclenché (patience={EARLY_STOPPING_PATIENCE}).")
        break

print(f"Best val macro-F1: {best_val_f1:.4f} (epoch {best_epoch})")

if best_state is not None:
    model.load_state_dict(best_state)

save_path = "best_efficientnet_b2_plankton_fast.pt" if FAST_MODE else "best_efficientnet_b2_plankton_full.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "class_to_idx": class_to_idx,
        "img_size": IMG_SIZE,
        "best_val_macro_f1": best_val_f1,
        "best_epoch": best_epoch,
        "fast_mode": FAST_MODE,
    },
    save_path,
 )
print(f"Modèle sauvegardé: {save_path}")

Device: cuda


c:\Users\thiev\OneDrive\Bureau\Plankton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train subset: 984,444 | Val subset: 109,341


Epoch 01/12 | train_loss=0.5348 | val_loss=1.0595 | val_macro_f1=0.5024


Epoch 02/12 | train_loss=0.3067 | val_loss=0.8573 | val_macro_f1=0.5514


Epoch 03/12 | train_loss=0.2498 | val_loss=0.7963 | val_macro_f1=0.5631


... (5 outputs left out)

Epoch 09/12 | train_loss=0.1078 | val_loss=0.6938 | val_macro_f1=0.6495


Epoch 10/12 | train_loss=0.0961 | val_loss=0.7087 | val_macro_f1=0.6511


Epoch 11/12 | train_loss=0.0888 | val_loss=0.6995 | val_macro_f1=0.6656


Epoch 12/12 | train_loss=0.0848 | val_loss=0.6886 | val_macro_f1=0.6673
Best val macro-F1: 0.6673 (epoch 12)
Modèle sauvegardé: best_efficientnet_b2_plankton_full.pt


In [6]:
# Cellule 9 — Génération du fichier de soumission Kaggle (imgname,label)
import csv
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from tqdm.auto import tqdm

assert "test_ds" in globals(), "Exécute d'abord la cellule 7 (chargement datasets)."
assert "class_to_idx" in globals(), "Exécute d'abord la cellule 7 (chargement datasets)."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    raise RuntimeError("GPU non disponible. Vérifie le kernel et CUDA.")

torch.backends.cudnn.benchmark = True
num_classes = len(class_to_idx)

# Recrée la même architecture qu'à l'entraînement (EfficientNet-B2)
infer_model = models.efficientnet_b2(weights=None)
in_features = infer_model.classifier[1].in_features
infer_model.classifier[1] = nn.Linear(in_features, num_classes)

# Priorité au checkpoint full B2, puis fallback
candidate_ckpts = [
    Path("best_efficientnet_b2_plankton_full.pt"),
    Path("best_efficientnet_b2_plankton_fast.pt"),
    Path("best_efficientnet_b0_plankton_full.pt"),
    Path("best_efficientnet_b0_plankton_fast.pt"),
    Path("best_efficientnet_b0_plankton.pt"),
]
ckpt_path = next((p for p in candidate_ckpts if p.exists()), None)
if ckpt_path is None:
    raise FileNotFoundError("Aucun checkpoint trouvé. Entraîne d'abord le modèle (cellule 8).")

checkpoint = torch.load(ckpt_path, map_location="cpu")
infer_model.load_state_dict(checkpoint["state_dict"])
infer_model = infer_model.to(device)
infer_model.eval()
print(f"Checkpoint chargé: {ckpt_path}")

# Sur Windows notebook, num_workers=0 évite le blocage/démarrage très lent des workers
BATCH_SIZE_TEST = 256
NUM_WORKERS_TEST = 0

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE_TEST,
    shuffle=False,
    num_workers=NUM_WORKERS_TEST,
    pin_memory=True,
)

rows = []
with torch.inference_mode():
    test_bar = tqdm(test_loader, desc="Inference test", total=len(test_loader), leave=True)
    for batch in test_bar:
        if len(batch) == 2:
            images, imgnames = batch
        elif len(batch) == 3:
            images, imgnames, _ = batch
        else:
            raise RuntimeError("Format inattendu du batch test.")

        images = images.to(device, non_blocking=True)
        logits = infer_model(images)
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        rows.extend(zip(imgnames, preds))

# Garantit l'ordre numérique des noms (0.jpg, 1.jpg, ..., 121427.jpg)
def _csv_sort_key(item):
    name = item[0]
    stem = Path(name).stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)

rows = sorted(rows, key=_csv_sort_key)

submission_path = Path("submission.csv")
with submission_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["imgname", "label"])
    for imgname, label in rows:
        writer.writerow([imgname, int(label)])

print(f"Fichier généré: {submission_path.resolve()}")
print("Aperçu:")
for imgname, label in rows[:5]:
    print(f"{imgname},{int(label)}")

Device: cuda
Checkpoint chargé: best_efficientnet_b2_plankton_full.pt


Inference test: 100%|██████████| 475/475 [06:09<00:00,  1.29it/s]


Fichier généré: C:\Users\thiev\OneDrive\Bureau\Plankton\submission.csv
Aperçu:
0.jpg,33
1.jpg,33
2.jpg,70
3.jpg,13
4.jpg,4
